In [ ]:
# Cell 1: Environment Setup and Library Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# Scikit-Learn Imports
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    mean_squared_error, r2_score, mean_absolute_error,
    accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
)

# Set plotting style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

print("All libraries imported successfully!")

In [ ]:
# Cell 2: Project 1 - Student Performance Prediction Data Generation
np.random.seed(42)
n_students = 500

attendance = np.random.uniform(50, 100, n_students)
internal_marks = np.random.uniform(10, 30, n_students)
assignments = np.random.uniform(5, 20, n_students)
study_hours = np.random.uniform(1, 10, n_students)
prev_sem_gpa = np.random.uniform(4.0, 10.0, n_students)

# Target: Classification (Grade: 1 for High Performer, 0 for Needs Support)
score = (0.25 * attendance) + (1.2 * internal_marks) + (1.0 * assignments) + (2.0 * study_hours) + (3.0 * prev_sem_gpa)
grade_category = (score > 80).astype(int)

df_student = pd.DataFrame({
    'Attendance_Pct': np.round(attendance, 2),
    'Internal_Marks': np.round(internal_marks, 2),
    'Assignment_Score': np.round(assignments, 2),
    'Study_Hours_Per_Week': np.round(study_hours, 2),
    'Prev_Sem_GPA': np.round(prev_sem_gpa, 2),
    'Performance_Class': grade_category
})

# Save dataset to CSV
df_student.to_csv("student_performance_data.csv", index=False)
print("Project 1 Dataset Shape:", df_student.shape)
df_student.head()

In [ ]:
# Cell 3: Project 1 - Model Training & Evaluation (Classification)
X_student = df_student.drop(columns=['Performance_Class'])
y_student = df_student['Performance_Class']

# Train-Test Split
X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
    X_student, y_student, test_size=0.2, random_state=42, stratify=y_student
)

# Standard Scaling Pipeline
scaler = StandardScaler()
X_train_s_scaled = scaler.fit_transform(X_train_s)
X_test_s_scaled = scaler.transform(X_test_s)

# Initialize Models
models_student = {
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42)
}

results_student = {}

for name, model in models_student.items():
    model.fit(X_train_s_scaled, y_train_s)
    y_pred = model.predict(X_test_s_scaled)
    
    acc = accuracy_score(y_test_s, y_pred)
    prec = precision_score(y_test_s, y_pred)
    rec = recall_score(y_test_s, y_pred)
    f1 = f1_score(y_test_s, y_pred)
    
    results_student[name] = {
        'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'F1-Score': f1, 'Model': model
    }

# Display Results Table
df_res_student = pd.DataFrame(results_student).T[['Accuracy', 'Precision', 'Recall', 'F1-Score']]
print("--- Project 1 Performance Comparison ---")
print(df_res_student)

In [ ]:
# Cell 4: Project 1 - Confusion Matrix Visualization & Model Saving
best_student_model_name = df_res_student['F1-Score'].idxmax()
best_student_model = results_student[best_student_model_name]['Model']

# Plot Confusion Matrix
y_pred_best = best_student_model.predict(X_test_s_scaled)
cm = confusion_matrix(y_test_s, y_pred_best)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Needs Support', 'High Performer'], yticklabels=['Needs Support', 'High Performer'])
plt.title(f'Confusion Matrix: {best_student_model_name}')
plt.xlabel('Predicted Class')
plt.ylabel('Actual Class')
plt.tight_layout()
plt.show()

# Save Scaler & Best Model
joblib.dump(best_student_model, "best_student_model.pkl")
joblib.dump(scaler, "student_scaler.pkl")
print(f"Saved {best_student_model_name} as 'best_student_model.pkl'")